
# 25 — Direct Preference Optimization

## 01. Deriving DPO from KL-Regularized RLHF

### Motivation

RLHF traditionally learns a reward model and optimizes
a policy using reinforcement learning.

DPO directly optimizes the policy from pairwise preference
data, using a fixed reference policy.

From KL-regularized reward maximization:

$$
\pi^*(y|x)
=
\frac{1}{Z(x)}
\pi_{\mathrm{ref}}(y|x)e^{r(x,y)/\beta}
$$

We recover the reward representation:

$$
r(x,y)
=
\beta\log
\frac{\pi^*(y|x)}{\pi_{\mathrm{ref}}(y|x)}
+
\beta\log Z(x)
$$

The prompt-dependent normalization constant cancels in
the Bradley-Terry reward difference.

Replacing the optimal policy with our trainable policy
gives the DPO objective:

$$
\mathcal L_{\mathrm{DPO}}
=
-\mathbb E_{(x,y_w,y_l)}
\left[
\log\sigma\left(
\beta
\left[
\log\frac{\pi_\theta(y_w|x)}{\pi_{\mathrm{ref}}(y_w|x)}
-
\log\frac{\pi_\theta(y_l|x)}{\pi_{\mathrm{ref}}(y_l|x)}
\right]
\right)
\right]
$$

The objective uses complete-response log-probabilities,
not isolated token probabilities.


In [ ]:
import torch
import torch.nn.functional as F

beta = 0.5

# (Y=2,): reference probabilities [chosen, rejected]
reference_probs = torch.tensor([0.05, 0.95], dtype=torch.float64)

# (Y=2,): current policy probabilities [chosen, rejected]
policy_probs = torch.tensor([0.20, 0.80], dtype=torch.float64)

# (Y=2,): log-probability improvement relative to reference
log_ratios = policy_probs.log() - reference_probs.log()

# (): chosen-minus-rejected relative improvement
relative_margin = log_ratios[0] - log_ratios[1]

# (): Bradley-Terry logit under the DPO reward parameterization
dpo_logit = beta * relative_margin

# (): DPO negative log-likelihood
dpo_loss = -F.logsigmoid(dpo_logit)

# (): implicit preference probability
preference_probability = torch.sigmoid(dpo_logit)

print("Relative margin:", relative_margin.item())
print("DPO preference probability:", preference_probability.item())
print("DPO loss:", dpo_loss.item())

# A key distinction: chosen can remain less likely in the policy,
# while its improvement relative to reference is greater.
assert policy_probs[0] < policy_probs[1]
assert relative_margin > 0

print("Relative preference improvement verified.")


Relative margin: 1.5581446180465497
DPO preference probability: 0.6854801408612435
DPO loss: 0.3776357506966529
Relative preference improvement verified.



### Takeaways

DPO starts from the optimal policy of KL-regularized
reward maximization.

The reward can be represented through a policy-to-reference
log-probability ratio, up to a prompt-dependent constant.

The constant cancels when comparing two responses to the
same prompt using the Bradley-Terry model.

DPO trains a policy directly from chosen/rejected preferences.

It does not require a separately trained reward model,
a trainable Critic, or PPO rollout collection.

The reference policy remains fixed.

DPO uses relative improvements in complete-response
log-probabilities, not merely raw response likelihoods.

Next: compute DPO gradients and apply the objective to
our actual GPT policies.



## 02. DPO Gradients and GPT Log-Probabilities

### Why does DPO produce a learning signal?

Let

$$
z=\beta[
(\log\pi_\theta(y_w|x)-\log\pi_\theta(y_l|x))
-
(\log\pi_{\mathrm{ref}}(y_w|x)-\log\pi_{\mathrm{ref}}(y_l|x))
]
$$

The DPO objective is:

$$
L=-\log\sigma(z)
$$

Its derivatives with respect to the current-policy
sequence log-probabilities are:

$$
\frac{\partial L}{\partial\log\pi_\theta(y_w|x)}
=-\beta\sigma(-z)
$$

$$
\frac{\partial L}{\partial\log\pi_\theta(y_l|x)}
=+\beta\sigma(-z)
$$

These derivatives encourage an increase in the relative
chosen-minus-rejected margin.

At initialization, when the actor equals the reference:

$$
z=0,\quad L=\log2
$$

The gradient remains nonzero.

For a neural language model, the two sequence log-probabilities
are functions of shared model parameters. Their absolute
changes after an optimizer step are not independently guaranteed.


In [2]:
dpo_beta = 0.5

# (Y=2,): frozen reference log-probs [chosen, rejected]
reference_logps = torch.tensor([0.05, 0.95], dtype=torch.float64).log()

# (Y=2,): independently differentiable scores for this math check
actor_logps = reference_logps.clone().detach().requires_grad_(True)

# (): relative chosen-over-rejected margin
margin = (actor_logps[0] - actor_logps[1]) - (reference_logps[0] - reference_logps[1])

# (): DPO objective
z = dpo_beta * margin
loss = -F.logsigmoid(z)

# (Y=2,): gradients with respect to the two log-prob scores
gradients = torch.autograd.grad(loss, actor_logps)[0]

# (Y=2,): gradients derived analytically
expected_gradients = torch.tensor([-dpo_beta / 2, dpo_beta / 2], dtype=torch.float64)

torch.testing.assert_close(gradients, expected_gradients)

print("Initial DPO loss:", loss.item())
print("Chosen gradient:", gradients[0].item())
print("Rejected gradient:", gradients[1].item())


Initial DPO loss: 0.6931471805599453
Chosen gradient: -0.25
Rejected gradient: 0.25


In [3]:
import copy
from pathlib import Path

from llmfp.data import (
    CharacterTokenizer,
    pad_supervised_batch,
    shift_supervised_tokens,
)
from llmfp.models import GPT, GPTConfig
from llmfp.training import completion_sequence_log_probs

checkpoint = torch.load(
    Path("../checkpoints/tiny_shakespeare_sft/step_00040.pt"),
    map_location="cpu",
    weights_only=True,
)

tokenizer = CharacterTokenizer(checkpoint["tokenizer_vocabulary"])
config = GPTConfig(**checkpoint["model_config"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Frozen SFT reference.
dpo_reference = GPT(config).to(device)
dpo_reference.load_state_dict(checkpoint["model_state_dict"])
dpo_reference.requires_grad_(False)
dpo_reference.eval()

# Trainable Actor initialized identically.
dpo_actor = copy.deepcopy(dpo_reference)
dpo_actor.requires_grad_(True)
dpo_actor.eval()

# Same three preference pairs as Lesson 23.
preference_pairs = [
    ("Say yes", "Yes", "No"),
    ("Say no", "No", "Yes"),
    ("Say hello", "Hello", "Farewell"),
]

encoded_candidates = []

for prompt, chosen, rejected in preference_pairs:
    prefix = f"User: {prompt}\nAssistant: "
    prefix_length = len(tokenizer.encode(prefix))

    # Interleaved order: chosen, rejected.
    for response in (chosen, rejected):
        full_ids = torch.tensor(
            tokenizer.encode(prefix + response + "\n"), dtype=torch.long
        )

        # (L,): only completion characters contribute to logP
        target_mask = torch.zeros_like(full_ids, dtype=torch.bool)
        target_mask[prefix_length:] = True

        encoded_candidates.append(shift_supervised_tokens(full_ids, target_mask))

# (B=6, T_max): causal predictor inputs and completion-only labels
dpo_inputs, dpo_labels = pad_supervised_batch(
    encoded_candidates, pad_token_id=tokenizer.token_to_id[" "]
)

dpo_inputs = dpo_inputs.to(device)
dpo_labels = dpo_labels.to(device)

print("Preference pairs:", len(preference_pairs))
print("Candidate batch:", dpo_inputs.shape)
print("Device:", device)


Preference pairs: 3
Candidate batch: torch.Size([6, 35])
Device: cuda


In [4]:
# (B=6,): frozen reference sequence log-probabilities
with torch.no_grad():
    reference_logits = dpo_reference(dpo_inputs)
    reference_sequence_logps = completion_sequence_log_probs(
        reference_logits, dpo_labels
    )

# (B=6,): differentiable Actor sequence log-probabilities
actor_logits = dpo_actor(dpo_inputs)
actor_sequence_logps = completion_sequence_log_probs(actor_logits, dpo_labels)

# (N_pairs=3,): chosen-minus-rejected sequence margins
actor_margins = actor_sequence_logps[0::2] - actor_sequence_logps[1::2]
reference_margins = reference_sequence_logps[0::2] - reference_sequence_logps[1::2]

# (N_pairs=3,): DPO preference logits
dpo_logits = dpo_beta * (actor_margins - reference_margins)

# (): batch-mean DPO loss
dpo_gpt_loss = -F.logsigmoid(dpo_logits).mean()

dpo_actor.zero_grad(set_to_none=True)
dpo_gpt_loss.backward()

# (): gradient norm of the GPT output projection
lm_head_grad_norm = dpo_actor.lm_head.weight.grad.norm()

torch.testing.assert_close(
    dpo_logits.detach(), torch.zeros_like(dpo_logits), atol=1e-5, rtol=0
)

assert lm_head_grad_norm.item() > 0

print("Reference preference margins:", reference_margins.tolist())
print("DPO initial loss:", dpo_gpt_loss.item())
print("GPT LM-head gradient norm:", lm_head_grad_norm.item())
print("Reference trainable:", any(p.requires_grad for p in dpo_reference.parameters()))


Reference preference margins: [20.45224952697754, -18.623489379882812, 10.309772491455078]
DPO initial loss: 0.6931471824645996
GPT LM-head gradient norm: 2.299677848815918
Reference trainable: False



### DPO Gradients — Takeaways

The DPO gradient is nonzero even when the trainable actor
and frozen reference have identical parameters.

The chosen and rejected terms contribute opposite signs
to the loss gradient with respect to sequence log-probabilities.

The optimized quantity is a relative preference margin,
not two independently controllable probabilities.

We restored the SFT checkpoint into a trainable GPT actor
and a frozen reference policy.

We reused the same three preference pairs and the existing
completion_sequence_log_probs implementation from Lesson 23.

The initial DPO logits are zero and the initial loss is log(2).

Backpropagation produces nonzero gradients in the GPT actor.
No optimizer update has been performed yet.

Next: train the policy directly with the DPO objective.



## 03. Training the GPT Policy with DPO

We now optimize GPT parameters directly using preference pairs.

For each pair:

$$
\Delta_\theta =
(\log\pi_\theta(y_w|x)-\log\pi_\theta(y_l|x))
-
(\log\pi_{\mathrm{ref}}(y_w|x)-\log\pi_{\mathrm{ref}}(y_l|x))
$$

The objective is:

$$
L_{\mathrm{DPO}}
=
-\mathbb E[\log\sigma(\beta\Delta_\theta)]
$$

A positive relative improvement does not necessarily mean
the chosen response is already more likely than the rejected one.

We will train on the same three preference pairs, keeping
the reference policy frozen throughout optimization.


In [ ]:
# Start from the frozen SFT reference.
dpo_actor = copy.deepcopy(dpo_reference)
dpo_actor.requires_grad_(True)
dpo_actor.eval()

dpo_optimizer = torch.optim.AdamW(dpo_actor.parameters(), lr=1e-5, weight_decay=0.0)

train_steps = 40
loss_history = []

# (N_pairs=3,): fixed reference margins from Section 02
fixed_reference_margins = reference_margins.detach()

# (): expected DPO loss before any optimizer update
initial_dpo_loss = torch.log(torch.tensor(2.0)).item()

for step in range(1, train_steps + 1):
    # (B=6, T_max, V): current GPT predictions
    logits = dpo_actor(dpo_inputs)

    # (B=6,): differentiable completion sequence logP
    logps = completion_sequence_log_probs(logits, dpo_labels)

    # (N_pairs=3,): chosen-minus-rejected current margins
    current_margins = logps[0::2] - logps[1::2]

    # (N_pairs=3,): reference-corrected preference logits
    preference_logits = dpo_beta * (current_margins - fixed_reference_margins)

    # (): differentiable mean DPO objective
    loss = -F.logsigmoid(preference_logits).mean()

    dpo_optimizer.zero_grad(set_to_none=True)
    loss.backward()
    dpo_optimizer.step()

    loss_history.append(loss.item())

    if step in (1, 10, 20, 40):
        print(f"Step {step:02d} | DPO loss = {loss.item():.4f}")

print("DPO training completed.")


Step 01 | DPO loss = 0.6931
Step 10 | DPO loss = 0.1327
Step 20 | DPO loss = 0.0364
Step 40 | DPO loss = 0.0154
DPO training completed.


In [6]:
with torch.no_grad():
    # (B=6,): updated Actor sequence log-probabilities
    final_logps = completion_sequence_log_probs(dpo_actor(dpo_inputs), dpo_labels)

    # (N_pairs=3,): updated chosen-minus-rejected margins
    final_margins = final_logps[0::2] - final_logps[1::2]

    # (N_pairs=3,): improvement relative to frozen SFT margins
    margin_improvement = final_margins - fixed_reference_margins

    # (N_pairs=3,): DPO preference probabilities
    dpo_preference_probs = torch.sigmoid(dpo_beta * margin_improvement)

    # (): final DPO objective
    final_dpo_loss = -F.logsigmoid(dpo_beta * margin_improvement).mean()

for index, (prompt, chosen, rejected) in enumerate(preference_pairs):
    print(
        f"{prompt:12s} | "
        f"reference={fixed_reference_margins[index].item():+8.3f} | "
        f"current={final_margins[index].item():+8.3f} | "
        f"improvement={margin_improvement[index].item():+8.3f} | "
        f"P_DPO={dpo_preference_probs[index].item():.3f}"
    )

assert torch.isfinite(final_dpo_loss)
assert final_dpo_loss.item() < initial_dpo_loss

# Verify that only the Actor was optimized.
assert not any(parameter.requires_grad for parameter in dpo_reference.parameters())

print("Initial DPO loss:", initial_dpo_loss)
print("Final DPO loss:", final_dpo_loss.item())
print("DPO optimization check passed.")


Say yes      | reference= +20.452 | current= +32.641 | improvement= +12.189 | P_DPO=0.998
Say no       | reference= -18.623 | current=  -6.752 | improvement= +11.871 | P_DPO=0.997
Say hello    | reference= +10.310 | current= +16.703 | improvement=  +6.393 | P_DPO=0.961
Initial DPO loss: 0.6931471824645996
Final DPO loss: 0.01499272882938385
DPO optimization check passed.



### DPO Training — Takeaways

We trained a GPT policy directly using chosen/rejected
preference pairs and the DPO objective.

The frozen SFT reference provided fixed sequence-level
log-probability margins.

Each optimizer step updated the current Actor using
reference-corrected preference margins.

DPO optimizes relative preference improvement, not
necessarily the absolute ranking of responses.

A lower DPO loss shows better fitting of the training
preference pairs, not necessarily better generation.

The DPO gradient decreases as the relative preference
margin becomes strongly positive.

Next: evaluate actual generated responses and compare
DPO against the original SFT reference.



## 04. DPO Evaluation

A lower DPO loss does not necessarily imply better generation.

We test two questions:

1. Does the trained policy generate different responses?
2. Do preference-margin improvements transfer to pairs excluded
   from DPO training?

For a preference pair, define:

$$
m_\theta =
\log\pi_\theta(y_w|x)-\log\pi_\theta(y_l|x)
$$

and

$$
\Delta m = m_\theta-m_{\mathrm{ref}}
$$

A positive relative improvement does not guarantee that
the raw margin becomes positive.

Greedy decoding also differs from comparing two complete
response probabilities: greedy selects the highest-probability
token at each successive step.


In [7]:
from llmfp.generation import generate_greedy_cached


@torch.no_grad()
def greedy_response(
    model: GPT, instruction: str, max_new_tokens: int = 24
) -> tuple[str, bool]:
    """Generate the first response line from a GPT policy."""
    prompt = f"User: {instruction}\nAssistant: "

    # (B=1, P): prompt token IDs
    prompt_ids = torch.tensor(
        [tokenizer.encode(prompt)], dtype=torch.long, device=device
    )

    # (B=1, P+T): prompt and generated tokens
    generated = generate_greedy_cached(model, prompt_ids, max_new_tokens=max_new_tokens)

    completion = tokenizer.decode(generated[0, prompt_ids.shape[1] :].cpu().tolist())

    return completion.split("\n", 1)[0], "\n" in completion


evaluation_cases = [
    ("Train", "Say yes", "Yes"),
    ("Train", "Say no", "No"),
    ("Train", "Say hello", "Hello"),
    ("DPO held-out", "Say welcome", "Welcome"),
    ("DPO held-out", "Say please", "Please"),
    ("DPO held-out", "Say goodbye", "Goodbye"),
]

greedy_results = []

for split, prompt, expected in evaluation_cases:
    ref_answer, ref_ended = greedy_response(dpo_reference, prompt)
    dpo_answer, dpo_ended = greedy_response(dpo_actor, prompt)

    greedy_results.append({
        "prompt": prompt,
        "reference": ref_answer,
        "dpo": dpo_answer,
        "reference_exact": ref_ended and ref_answer == expected,
        "dpo_exact": dpo_ended and dpo_answer == expected,
    })

    print(
        f"{prompt:12s} | "
        f"SFT={ref_answer!r:12s} | "
        f"DPO={dpo_answer!r:12s} | "
        f"expected={expected!r} | "
        f"changed={ref_answer != dpo_answer}"
    )

print(
    "Reference exact:",
    sum(row["reference_exact"] for row in greedy_results),
    "/",
    len(greedy_results),
)

print(
    "DPO exact:",
    sum(row["dpo_exact"] for row in greedy_results),
    "/",
    len(greedy_results),
)


Say yes      | SFT='Yes'        | DPO='Yes'        | expected='Yes' | changed=False
Say no       | SFT='Yes'        | DPO='Thanks'     | expected='No' | changed=True
Say hello    | SFT='Hello'      | DPO='Hello'      | expected='Hello' | changed=False
Say welcome  | SFT='Thanks'     | DPO='Thanks'     | expected='Welcome' | changed=False
Say please   | SFT='Farewell'   | DPO='Hello'      | expected='Please' | changed=True
Say goodbye  | SFT='Thanks'     | DPO='Thanks'     | expected='Goodbye' | changed=False
Reference exact: 2 / 6
DPO exact: 2 / 6


In [8]:
heldout_pairs = [
    ("Say welcome", "Welcome", "Yes"),
    ("Say please", "Please", "No"),
    ("Say goodbye", "Goodbye", "Hello"),
]

heldout_examples = []

for prompt, chosen, rejected in heldout_pairs:
    prefix = f"User: {prompt}\nAssistant: "
    prefix_length = len(tokenizer.encode(prefix))

    for response in (chosen, rejected):
        # (L_i,): full prompt and completion
        full_ids = torch.tensor(
            tokenizer.encode(prefix + response + "\n"), dtype=torch.long
        )

        # (L_i,): completion-only target mask
        target_mask = torch.zeros_like(full_ids, dtype=torch.bool)
        target_mask[prefix_length:] = True

        heldout_examples.append(shift_supervised_tokens(full_ids, target_mask))

# (B=6, T_max): padded predictor inputs and labels
heldout_inputs, heldout_labels = pad_supervised_batch(
    heldout_examples,
    pad_token_id=tokenizer.token_to_id[" "],
)

heldout_inputs = heldout_inputs.to(device)
heldout_labels = heldout_labels.to(device)

with torch.no_grad():
    # (B=6,): frozen reference sequence logP
    ref_logps = completion_sequence_log_probs(
        dpo_reference(heldout_inputs), heldout_labels
    )

    # (B=6,): trained DPO Actor sequence logP
    actor_logps = completion_sequence_log_probs(
        dpo_actor(heldout_inputs), heldout_labels
    )

    # (N_pairs=3,): raw chosen-minus-rejected margins
    ref_margins = ref_logps[0::2] - ref_logps[1::2]
    actor_margins = actor_logps[0::2] - actor_logps[1::2]

    # (N_pairs=3,): relative preference improvement
    heldout_improvements = actor_margins - ref_margins

for index, (prompt, chosen, rejected) in enumerate(heldout_pairs):
    print(
        f"{prompt:12s} | "
        f"SFT margin={ref_margins[index].item():+8.3f} | "
        f"DPO margin={actor_margins[index].item():+8.3f} | "
        f"improvement={heldout_improvements[index].item():+8.3f}"
    )

assert torch.isfinite(heldout_improvements).all()


Say welcome  | SFT margin= -18.617 | DPO margin= -18.646 | improvement=  -0.029
Say please   | SFT margin=  -3.268 | DPO margin=  -4.480 | improvement=  -1.212
Say goodbye  | SFT margin= -32.329 | DPO margin= -35.934 | improvement=  -3.604



### DPO Evaluation — Takeaways

DPO improves relative preference margins on training pairs,
but this does not necessarily change generated responses.

Greedy decoding operates token by token and is not equivalent
to comparing two complete-response probabilities.

Held-out preference pairs provide a small diagnostic of
whether training improvements transfer to other prompts.

A positive reference-relative margin improvement does not
necessarily imply a positive absolute chosen/rejected margin.

Training preference fitting and actual generation quality
must be evaluated separately.

No additional optimization was performed during evaluation.
